In [1]:
from sys import platform
from pathlib import Path
import os
import numpy as np
import imageio.v3 as iio
import cv2
import matplotlib.pyplot as plt
from IPython.display import Audio

from openneuro.mri_singledir import save_path

In [2]:
def _search_blk_swath(arr, value=1, search_n=10):
    """Search <value> consecutive black rows to determine row pixel to mask image."""
    c = 0
    l = len(arr)
    i = 0
    while c < search_n and i < len(arr)-1:
        if arr[i] == value:
            c += 1
            if arr[i+1] != value:
                c = 0 # Reset counter
            i += 1
        else:
            i+=1
    return i


def remove_noise_blk(clean_img, value=1, search_n=None, blk_thresh=None):
    """We want to remove all noise in black portion of lower half of each image"""
    height, _ = clean_img.shape[:2]
    row_means_r = np.mean(clean_img[:, :, 0], axis=1) # Row average RGB value for R channel
    black_rows = (row_means_r <= blk_thresh).astype(int) # Pixel rows with <= blk_thresh R value (probably black) 
    mid_row = int(np.median(range(height))) # Leveraging the fact that all frames have non-black in the median pixel row
    bttm_blk_rows = black_rows[mid_row:]
    bttm_blk_start = _search_blk_swath(bttm_blk_rows, value=value, search_n=search_n)
    global_blk_start = mid_row + bttm_blk_start # Add upper-half rows removed above
    return global_blk_start


def plot_mri(img, save_img=False, save_path=None, grays=True):
    if grays:
        plt.imshow(img, cmap='Grays_r')
    else:
        plt.imshow(img)
    plt.axis('off')
    plt.margins(x=0)
    plt.subplots_adjust(top=1, bottom=0, right=1, left=0, hspace=0, wspace=0)
    if save_img:
        assert(save_path is not None), 'No path provided'
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    else:
        plt.show()
    plt.close()


#### System Paths

In [3]:
if platform == 'linux':
    home_dir = os.path.expanduser('~')
elif platform == 'win32':
    home_dir = os.path.expandvars(r'%HOMEDRIVE%%HOMEPATH%\OneDrive') #Stupid OneDrive
else:
    raise ValueError("Cannot set home directory for this OS.")
print(home_dir)

/home/monorhesus


#### Body sex

In [4]:
sex = 'Male' # Male, Female

#### Output directories

In [5]:
img_dir = os.path.join('Pictures', 'visible-human')
img_dir

'Pictures/visible-human'

In [7]:
# Working directory
proj_dir = os.path.join(home_dir, img_dir) # As in ./human-visibility-dl.ipynb

# Raw directory
raw_dir = os.path.join(proj_dir, sex.lower(),'raw') # Images from ./human-visibility-dl.ipynb

# Clean directory
clean_dir = os.path.join(proj_dir, sex.lower(), 'clean')
mk_clean_dir = Path(clean_dir)
mk_clean_dir.mkdir(parents=True, exist_ok=True)

print(f'Raw dir: {raw_dir}')
print(f'Clean dir: {clean_dir}')

Raw dir: /home/monorhesus/Pictures/visible-human/male/raw
Clean dir: /home/monorhesus/Pictures/visible-human/male/clean


#### Get image paths

##### Select body part (all, abdomen, thorax, head, pelvis, thighs, legs) 

In [8]:
body_part = 'all'

In [9]:
extension = '.png'
raw_imgs = []
for root, dirs, files in os.walk(raw_dir):
    if body_part != 'all':
        dirs[:] = [d for d in dirs if d == body_part]
    for file in files:
        if file.endswith(extension):
            raw_imgs.append(os.path.join(root, file))
print(f'{len(raw_imgs)} images ready.')
# print(raw_imgs[:5])

2920 images ready.


#### Clean images (crop, mask)
Original shape (1216, 2048, 3)

##### Force 16:9 ratio

In [17]:
# New size: 16:9 ratio by const
y_start=44
y_end=1080
x_start=92 #149 - 20 -37
x_end=1933 # 1876 + 20+37

w = x_end-x_start
h = y_end-y_start
print(f'Width: {w}, Height: {h}, Ratio: {w/h}')

Width: 1841, Height: 1036, Ratio: 1.777027027027027


In [18]:
# Body part directory
# body_part_dir = rf'{clean_dir}\{body_part}'
body_part_dir = os.path.join(clean_dir, body_part)
temp_mk_dir = Path(body_part_dir)
temp_mk_dir.mkdir(parents=True, exist_ok=True)
print(f'Saving images in {body_part_dir}')

Saving images in /home/monorhesus/Pictures/visible-human/male/clean/all


Actual cleaning

In [25]:
c = 0
for img_path in raw_imgs:
    # save_path = rf'{body_part_dir}\{img_path.split(os.sep)[-1]}'
    save_path = os.path.join(body_part_dir, img_path.split(os.sep)[-1])
    img = cv2.imread(img_path)

    # Crops
    clean_img_temp = img[y_start:y_end, x_start:x_end]

    # img = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB)

    # Interpolate
    new_width = 1920
    new_height = 1080
    new_size = (new_width, new_height)
    clean_img = cv2.resize(clean_img_temp, new_size, interpolation=cv2.INTER_LANCZOS4)
    # clean_img = cv2.resize(clean_img_temp, new_size, interpolation=cv2.INTER_CUBIC)


    # Convert BGR to HSV (beter for masking)
    hsv = cv2.cvtColor(clean_img, cv2.COLOR_BGR2HSV)
    lower_blue = np.array([50, 0, 0]) # hue, saturation, brigthness values from 
    upper_blue = np.array([179, 255, 255])
    mask = cv2.inRange(hsv, lower_blue, upper_blue)
  
    # Blur mask
    kernel_size = 1
    feathered_mask = cv2.GaussianBlur(mask, (kernel_size, kernel_size), 0)
    clean_img[feathered_mask > 0] = [0, 0, 0]
    
    # Remove nosie residuals in lower half of each image (remove image Id., grayscale, etc)
    height, width = clean_img.shape[:2]
    global_blk_start = remove_noise_blk(clean_img, value=1, search_n=3, blk_thresh=2)
    mask_height = height - global_blk_start
    clean_img[height - mask_height : height, 0 : width] = (0, 0, 0) # Slice bottomand set to black (0, 0, 0)

    cv2.imwrite(save_path, clean_img)
    med_img = np.median(range(len(raw_imgs)))
    if c == med_img:
        plot_mri(clean_img)

#### Animate segmentation
Takes a while and generates a 2.6Gb gif, see other script for other methods.

In [26]:
sorted_imgs = {}
for root, dirs, files in os.walk(body_part_dir):
    for file in files:
        k = int(''.join([c for c in file if c.isdigit()]))
        v = os.path.join(root, file)
        sorted_imgs[k] = v
sorted_imgs = {k: sorted_imgs[k] for k in sorted(sorted_imgs)} # Sort images 
[sorted_imgs[k] for k in list(sorted_imgs.keys())[:5]] # Verify proper order for axial segmentation

['/home/monorhesus/Pictures/visible-human/male/clean/all/a_vm1001.png',
 '/home/monorhesus/Pictures/visible-human/male/clean/all/a_vm1002.png',
 '/home/monorhesus/Pictures/visible-human/male/clean/all/a_vm1003.png',
 '/home/monorhesus/Pictures/visible-human/male/clean/all/a_vm1004.png',
 '/home/monorhesus/Pictures/visible-human/male/clean/all/a_vm1005.png']

In [27]:
clean_dir

'/home/monorhesus/Pictures/visible-human/male/clean'

In [16]:
filenames = [sorted_imgs[k] for k in sorted_imgs] 
images = [iio.imread(f) for f in filenames]
iio.imwrite(f'{clean_dir}\\gif-{sex}-{body_part}-axial.gif', images, duration=33.33, loop=0)
print('Done.')

Done.


In [30]:
if platform == 'linux':
    beep='/usr/share/sounds/freedesktop/stereo/bell.oga'
elif platform == 'win32':
    beep=r'C:\Windows\Media\tada.wav'
Audio(beep, autoplay=True)